# Santali vocabulary extension: tokenization audit and threshold comparison

**Purpose.** Add the measurements missing from the earlier notebook. No model is loaded and no training is run, so this notebook works on CPU.

1. Fertility before vs after extension on the exact same 2,000 sentences behind the original 1.442 figure.
2. Change in fertility and in average target-tokenized sentence length.
3. Audit of all 95 candidates: training frequency, original piece count, theoretical saving, and actual occurrences under the extension-aware encoder.
4. Theoretical savings, occurrence-based savings and measured savings are labelled separately.
5. A tokenization-only comparison across selection thresholds (100, 50, 20, 10, 5), using held-out validation text.

**Before you run:** attach the `common-tribal-preprocessing` dataset, add the `HF_TOKEN` secret, and run the install cell, then restart the session.

In [ ]:
# Cell 1: install pinned dependencies (same pins as the earlier notebook), then RESTART THE SESSION.
%pip install -q --no-cache-dir "transformers==4.46.3" "tokenizers==0.20.3" "IndicTransToolkit==1.1.1" sentencepiece sacrebleu matplotlib

In [2]:
import transformers
import sys
import torch

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("Transformers location:", transformers.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Transformers: 4.46.3
Transformers location: /usr/local/lib/python3.13/dist-packages/transformers/__init__.py
PyTorch: 2.11.0+cpu
CUDA available: False


In [1]:
# Cell 2: imports, Hugging Face login, output folder
import re, json, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

OUT_DIR = Path("/kaggle/working/vocab_audit")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT_DIR)

Output folder: /kaggle/working/vocab_audit


In [3]:
# Cell 3: load the frozen Santali splits (same files as before)
INPUT_ROOT = Path("/kaggle/input")
matches = [p for p in INPUT_ROOT.rglob("santali_train.parquet") if p.parent.name == "splits"]
assert len(matches) == 1, f"Expected one santali_train.parquet, found {len(matches)}"
SPLIT_DIR = matches[0].parent

train_df = pd.read_parquet(SPLIT_DIR / "santali_train.parquet")
val_df   = pd.read_parquet(SPLIT_DIR / "santali_validation.parquet")
test_df  = pd.read_parquet(SPLIT_DIR / "santali_test.parquet")

print("Split sizes (train / val / test):", len(train_df), len(val_df), len(test_df))
print("Expected from earlier work:      17883 / 993 / 994")
print("Columns:", train_df.columns.tolist())

Split sizes (train / val / test): 17883 993 994
Expected from earlier work:      17883 / 993 / 994
Columns: ['English', 'Santali']


In [4]:
# Cell 4: load the tokenizer only (no model needed)
from transformers import AutoTokenizer

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=hf_token)

sp = tok.tgt_spm
BASE_TARGET_VOCAB_SIZE = len(tok.tgt_encoder)
EOS_ID, UNK_ID = 2, 3

print("Tokenizer:", type(tok).__name__)
print("Target vocabulary entries:", BASE_TARGET_VOCAB_SIZE, "(earlier notebook: 122672)")
print("Target SentencePiece pieces:", sp.get_piece_size())

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Tokenizer: IndicTransTokenizer
Target vocabulary entries: 122672 (earlier notebook: 122672)
Target SentencePiece pieces: 128000


## Step 1: rebuild the candidate pool exactly as before
Same procedure as the earlier notebook: whitespace words from the **training** split only, no words made only of punctuation or symbols, more than one SentencePiece piece, frequency of at least 5, then drop forms containing punctuation or symbols. A reproduction check follows; the numbers should match the earlier results.

In [5]:
# Cell 5: word frequencies and candidate pool (training split only)
train_texts_full = train_df["Santali"].dropna().astype(str).tolist()

word_counts = Counter()
for t in train_texts_full:
    word_counts.update(t.split())
print("Unique whitespace words in training:", len(word_counts), "(earlier: 41757)")

def has_only_punct_symbols(word):
    return all(unicodedata.category(c)[0] in {"P", "S"} for c in word)

def is_word_only(text):
    return bool(text) and all(unicodedata.category(c)[0] not in {"P", "S"} for c in text)

rows = []
for word, freq in word_counts.items():
    if freq < 5 or has_only_punct_symbols(word):
        continue
    pieces = sp.encode(word, out_type=str)
    if len(pieces) > 1:
        rows.append({
            "word": word,
            "frequency": freq,
            "piece_count": len(pieces),
            "existing_pieces": " | ".join(pieces),
            "potential_piece_reduction": len(pieces) - 1,
        })

candidate_df = pd.DataFrame(rows)
clean_candidates = candidate_df[candidate_df["word"].map(is_word_only)].copy()

def select_candidates(min_freq):
    """Return (new_candidates, already_single_token) for a frequency threshold."""
    sub = clean_candidates[(clean_candidates["frequency"] >= min_freq) & (clean_candidates["piece_count"] > 1)]
    sub = (sub.drop_duplicates(subset=["word"])
              .sort_values(["frequency", "potential_piece_reduction", "word"],
                           ascending=[False, False, True], kind="stable")
              .reset_index(drop=True))
    existing = set(tok.tgt_encoder.keys())
    already = sub[sub["word"].isin(existing)].reset_index(drop=True)
    new = sub[~sub["word"].isin(existing)].reset_index(drop=True)
    return new, already

# Reproduction check against the earlier notebook (candidate types per threshold)
expected_types = {5: 3045, 10: 1232, 20: 451, 50: 97, 100: 21}
check = []
for thr, exp in expected_types.items():
    new, already = select_candidates(thr)
    got = len(new) + len(already)
    check.append({"min_freq": thr, "expected_types": exp, "rebuilt_types": got,
                  "match": got == exp, "new_tokens": len(new), "already_single_token": len(already)})
display(pd.DataFrame(check))
if not all(c["match"] for c in check):
    print("WARNING: candidate counts differ from the earlier notebook. Check the dataset version/splits before trusting later results.")

Unique whitespace words in training: 41757 (earlier: 41757)


,min_freq,expected_types,rebuilt_types,match,new_tokens,already_single_token
0,5,3045,3045,True,3020,25
1,10,1232,1232,True,1220,12
2,20,451,451,True,445,6
3,50,97,97,True,95,2
4,100,21,21,True,21,0


## Step 2: the extension-aware encoder (same logic as the one used in training)
This reproduces the final encoder from the earlier notebook (Section 10.18f plus the punctuation-aware pattern from 9.12): a candidate is replaced by its new ID only if it starts at the beginning of the text or after whitespace and is followed by end-of-text, whitespace or one of `. , ! ? ; : ( ) [ ] { } " '` or the danda marks. The text between matches is encoded with the original tokenizer.

In [6]:
# Cell 6: encoder factory
def encode_original_target_segment(segment):
    if not segment:
        return []
    with tok.as_target_tokenizer():
        pieces = tok.tokenize(segment)
        return tok.convert_tokens_to_ids(pieces)

def make_extension_encoder(words):
    """Build (encode_fn, word->new_id, compiled pattern) for an ordered list of new words."""
    cand_to_id = {w: BASE_TARGET_VOCAB_SIZE + i for i, w in enumerate(words)}
    pattern = None
    if words:
        ordered = sorted(words, key=lambda w: (-len(w), w))
        pattern = re.compile(
            r"(?<!\S)(" + "|".join(re.escape(w) for w in ordered) +
            r")(?=$|[\s\.,!?;:()\[\]{}\"'।॥])"
        )

    def encode(text, add_eos=False):
        ids, cursor = [], 0
        if pattern is not None:
            for m in pattern.finditer(text):
                ids.extend(encode_original_target_segment(text[cursor:m.start()]))
                ids.append(cand_to_id[m.group(1)])
                cursor = m.end()
        ids.extend(encode_original_target_segment(text[cursor:]))
        if add_eos and (not ids or ids[-1] != EOS_ID):
            ids.append(EOS_ID)
        return ids

    return encode, cand_to_id, pattern

# The 95-token extension used in the training experiment (threshold 50)
new_df_50, already_df_50 = select_candidates(50)
new_words_50 = new_df_50["word"].tolist()
ext_encode, candidate_to_id, ext_pattern = make_extension_encoder(new_words_50)

print("New tokens at threshold 50:", len(new_words_50), "(expected 95)")
print("Already single tokens (not added):", already_df_50["word"].tolist(), "(expected 2)")
print("Extended vocabulary size:", BASE_TARGET_VOCAB_SIZE + len(new_words_50))

# Integrity check: text with no candidate match must encode exactly like the original tokenizer
val_texts = val_df["Santali"].dropna().astype(str).tolist()
checked = mismatches = 0
for t in val_texts:
    if ext_pattern.search(t) is None:
        checked += 1
        if ext_encode(t) != encode_original_target_segment(t):
            mismatches += 1
print(f"Unchanged-text check on validation: {checked} sentences without candidates, {mismatches} mismatches (must be 0)")
assert mismatches == 0

New tokens at threshold 50: 95 (expected 95)
Already single tokens (not added): ['ᱨᱮᱧ', 'ᱨᱤᱧ'] (expected 2)
Extended vocabulary size: 122767
Unchanged-text check on validation: 672 sentences without candidates, 0 mismatches (must be 0)


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


## Step 3: fertility before vs after, on the exact same 2,000 sentences
The original 1.442 came from the first 2,000 non-null training targets (38,974 whitespace words, 56,205 SentencePiece pieces, 0 UNK). The same selection is used here. EOS is excluded on both sides, so the numbers are comparable with the earlier figure.

In [ ]:
# Cell 7: measurements 1 and 2
SAMPLE_N = 2000
sample_texts = train_df["Santali"].dropna().astype(str).head(SAMPLE_N).tolist()
n_sent = len(sample_texts)

sample_words = sum(len(t.split()) for t in sample_texts)

# (a) raw SentencePiece pieces: the method used for the original 1.442 figure
raw_pieces = sum(len(sp.encode(t, out_type=str)) for t in sample_texts)
print(f"Reproduction of the earlier baseline: words={sample_words} (expected 38974), "
      f"raw SP pieces={raw_pieces} (expected 56205), fertility={raw_pieces/sample_words:.3f} (expected 1.442)")

# (b) model-ID path, original tokenizer, and (c) extension-aware encoder
orig_ids = [encode_original_target_segment(t) for t in sample_texts]
ext_ids_sample = [ext_encode(t) for t in sample_texts]
orig_lens = np.array([len(x) for x in orig_ids])
ext_lens  = np.array([len(x) for x in ext_ids_sample])

orig_unk = sum(i == UNK_ID for ids in orig_ids for i in ids)
ext_unk  = sum(i == UNK_ID for ids in ext_ids_sample for i in ids)
new_tok_occ = sum(i >= BASE_TARGET_VOCAB_SIZE for ids in ext_ids_sample for i in ids)
sent_with_new = sum(any(i >= BASE_TARGET_VOCAB_SIZE for i in ids) for ids in ext_ids_sample)
sent_changed = int((orig_lens != ext_lens).sum())

def row(metric, a, b, fmt="{:.4f}"):
    return {"metric": metric, "original": fmt.format(a), "extended": fmt.format(b),
            "change": fmt.format(b - a), "change_%": f"{100*(b-a)/a:+.2f}%" if a else "n/a"}

fert_orig, fert_ext = orig_lens.sum()/sample_words, ext_lens.sum()/sample_words
summary_df = pd.DataFrame([
    row("total target tokens (no EOS)", orig_lens.sum(), ext_lens.sum(), "{:.0f}"),
    row("fertility (tokens per whitespace word)", fert_orig, fert_ext),
    row("avg tokens per sentence", orig_lens.mean(), ext_lens.mean(), "{:.3f}"),
    row("median tokens per sentence", np.median(orig_lens), np.median(ext_lens), "{:.1f}"),
])
display(summary_df)

print(f"\nWhitespace words (same denominator for both): {sample_words}")
print(f"Raw-SP piece count vs model-ID path (original): {raw_pieces} vs {orig_lens.sum()}  -> "
      f"{'identical' if raw_pieces == orig_lens.sum() else 'DIFFERENT, report which one is used'}")
print(f"UNK tokens: original {orig_unk}, extended {ext_unk}")
print(f"New-token occurrences in the sample: {new_tok_occ}  (share of extended tokens: {100*new_tok_occ/ext_lens.sum():.2f}%)")
print(f"Sentences containing at least one new token: {sent_with_new}/{n_sent}; sentences whose length changed: {sent_changed}")
summary_df.to_csv(OUT_DIR / "fertility_before_after_2000.csv", index=False)

## Step 4: audit all 95 candidates, with three clearly separated kinds of saving
* **THEORETICAL saving** = whitespace-word frequency x (original pieces - 1). It assumes every occurrence is replaced.
* **OCCURRENCE-BASED saving** = number of times the real encoder inserted the new ID x (original pieces - 1). It reflects the regex's punctuation behaviour.
* **MEASURED saving** = original token total minus extended token total from actually running both encoders. This is the real figure; the total is reported at the end of the table.

In [ ]:
# Cell 8: candidate audit
piece_cnt = dict(zip(new_df_50["word"], new_df_50["piece_count"]))
saved_per_occ = {w: c - 1 for w, c in piece_cnt.items()}
existing_pieces = dict(zip(new_df_50["word"], new_df_50["existing_pieces"]))

# encoder occurrences over the full training set (regex = the encoder's matching rule)
occ_full = Counter()
for t in train_texts_full:
    for m in ext_pattern.finditer(t):
        occ_full[m.group(1)] += 1

# encoder occurrences in the 2,000-sentence sample, counted from the actual inserted IDs
id_to_word = {v: k for k, v in candidate_to_id.items()}
occ_sample_ids = Counter(id_to_word[i] for ids in ext_ids_sample for i in ids if i >= BASE_TARGET_VOCAB_SIZE)
occ_sample_regex = Counter(m.group(1) for t in sample_texts for m in ext_pattern.finditer(t))
assert occ_sample_ids == occ_sample_regex, "Encoder output and regex matches disagree"

sample_word_counts = Counter(w for t in sample_texts for w in t.split())

audit_rows = []
for w in new_words_50:
    f_full = word_counts[w]
    audit_rows.append({
        "word": w,
        "new_id": candidate_to_id[w],
        "original_pieces": piece_cnt[w],
        "existing_pieces": existing_pieces[w],
        "train_frequency": f_full,
        "THEORETICAL_saving_full_train": f_full * saved_per_occ[w],
        "encoder_occurrences_full_train": occ_full[w],
        "match_ratio_full_train": round(occ_full[w] / f_full, 3),
        "THEORETICAL_saving_sample": sample_word_counts[w] * saved_per_occ[w],
        "encoder_occurrences_sample": occ_sample_ids[w],
        "OCCURRENCE_BASED_saving_sample": occ_sample_ids[w] * saved_per_occ[w],
    })
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_DIR / "candidate_audit_95.csv", index=False, encoding="utf-8-sig")
display(audit_df.drop(columns=["existing_pieces"]))

measured_saving_sample = int(orig_lens.sum() - ext_lens.sum())
totals = pd.DataFrame([
    {"quantity": "THEORETICAL saving, full training set (pieces)", "value": int(audit_df["THEORETICAL_saving_full_train"].sum())},
    {"quantity": "THEORETICAL saving, 2,000-sentence sample (pieces)", "value": int(audit_df["THEORETICAL_saving_sample"].sum())},
    {"quantity": "OCCURRENCE-BASED saving, 2,000-sentence sample (pieces)", "value": int(audit_df["OCCURRENCE_BASED_saving_sample"].sum())},
    {"quantity": "MEASURED saving, 2,000-sentence sample (pieces)", "value": measured_saving_sample},
])
display(totals)
print("Candidates never matched in the full training set:", int((audit_df["encoder_occurrences_full_train"] == 0).sum()))
print("Candidates with match_ratio < 1 (some occurrences not replaced):", int((audit_df["match_ratio_full_train"] < 1).sum()))
print("Candidates with match_ratio > 1 (also matched before punctuation, not counted in whitespace frequency):",
      int((audit_df["match_ratio_full_train"] > 1).sum()))
totals.to_csv(OUT_DIR / "saving_totals.csv", index=False)

In [ ]:
# Cell 9: why do theoretical, occurrence-based and measured savings differ?

# (a) Occurrences the encoder misses: tokens whose punctuation-stripped form is a candidate but that are not matched
def strip_edge_punct(tok_str):
    s, e = 0, len(tok_str)
    while s < e and unicodedata.category(tok_str[s])[0] in {"P", "S"}:
        s += 1
    while e > s and unicodedata.category(tok_str[e-1])[0] in {"P", "S"}:
        e -= 1
    return tok_str[s:e]

cand_set = set(new_words_50)
missed = Counter()
for surface, freq in word_counts.items():
    core = strip_edge_punct(surface)
    if core in cand_set and surface != core:
        m = ext_pattern.match(surface)
        if not (m and m.group(1) == core):
            missed[surface] += freq
print("Near-miss surface forms (candidate with attached punctuation) not replaced by the encoder:")
print("  distinct forms:", len(missed), " total occurrences:", sum(missed.values()))
display(pd.DataFrame(missed.most_common(15), columns=["surface_form", "occurrences_not_replaced"]))

# (b) Sentence-level agreement between expected and measured saving
exp_delta = np.array([sum(saved_per_occ[m.group(1)] for m in ext_pattern.finditer(t)) for t in sample_texts])
act_delta = orig_lens - ext_lens
diff = act_delta - exp_delta
print("\nSentence-level check (measured saving minus occurrence-based expected saving):")
print("  sentences where they agree:", int((diff == 0).sum()), "of", n_sent)
print("  sentences where measured < expected:", int((diff < 0).sum()), "| measured > expected:", int((diff > 0).sum()))
print("  total difference (pieces):", int(diff.sum()))
print("  If non-zero, splitting the sentence around a candidate changes how the neighbouring text is segmented;")
print("  the MEASURED figure is the one to report.")

mis = np.where(diff != 0)[0][:5]
for i in mis:
    print("\nExample sentence index", int(i), "| expected", int(exp_delta[i]), "| measured", int(act_delta[i]))
    print(sample_texts[i][:160])

## Step 5: compare different token counts (tokenization only, no training)
For each threshold the candidate list is selected from the **training** split only. Its effect is measured on the 2,000 training sample and on the full **validation** split (held out from selection). A candidate list that saves much less on validation than on training is overfitted to the training text. The smallest training frequency shows how many examples the rarest new embedding would get.

In [ ]:
# Cell 10: tokenization-only threshold sweep
THRESHOLDS = [100, 50, 20, 10, 5]

full_train_pieces = sum(len(sp.encode(t, out_type=str)) for t in train_texts_full)
val_words = sum(len(t.split()) for t in val_texts)
val_orig_lens = np.array([len(encode_original_target_segment(t)) for t in val_texts])

sweep_rows = []
sweep_details = {}
for thr in THRESHOLDS:
    new_df, already_df = select_candidates(thr)
    words = new_df["word"].tolist()
    enc, c2id, pat = make_extension_encoder(words)

    s_ids = [enc(t) for t in sample_texts]
    v_ids = [enc(t) for t in val_texts]
    s_len = np.array([len(x) for x in s_ids]); v_len = np.array([len(x) for x in v_ids])

    id_counts_val = Counter(i for ids in v_ids for i in ids if i >= BASE_TARGET_VOCAB_SIZE)
    theo = int((new_df["frequency"] * new_df["potential_piece_reduction"]).sum())

    sweep_rows.append({
        "min_freq": thr,
        "new_tokens": len(words),
        "THEORETICAL_saving_train_%": round(100 * theo / full_train_pieces, 2),
        "train_sample_fertility": round(s_len.sum() / sample_words, 4),
        "train_sample_MEASURED_reduction_%": round(100 * (orig_lens.sum() - s_len.sum()) / orig_lens.sum(), 2),
        "val_fertility_orig": round(val_orig_lens.sum() / val_words, 4),
        "val_fertility_ext": round(v_len.sum() / val_words, 4),
        "val_MEASURED_reduction_%": round(100 * (val_orig_lens.sum() - v_len.sum()) / val_orig_lens.sum(), 2),
        "val_new_token_share_%": round(100 * sum(id_counts_val.values()) / v_len.sum(), 2),
        "new_tokens_unused_in_val": len(words) - len(id_counts_val),
        "rarest_token_train_freq": int(new_df["frequency"].min()) if len(words) else 0,
    })
    sweep_details[thr] = {"words": words}
    print("done threshold", thr, "->", len(words), "new tokens")

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df)
sweep_df.to_csv(OUT_DIR / "threshold_sweep_tokenization.csv", index=False)
json.dump({str(k): v["words"] for k, v in sweep_details.items()},
          open(OUT_DIR / "candidate_lists_by_threshold.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

In [ ]:
# Cell 11: plot validation token reduction vs number of added tokens
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep_df["new_tokens"], sweep_df["val_MEASURED_reduction_%"], marker="o", label="validation (held out)")
ax.plot(sweep_df["new_tokens"], sweep_df["train_sample_MEASURED_reduction_%"], marker="s", label="train sample")
for _, r in sweep_df.iterrows():
    ax.annotate(f">={int(r['min_freq'])}", (r["new_tokens"], r["val_MEASURED_reduction_%"]),
                textcoords="offset points", xytext=(4, -12), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("number of added tokens (log scale)")
ax.set_ylabel("measured target-token reduction (%)")
ax.set_title("Token savings vs vocabulary size")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "token_reduction_vs_size.png", dpi=150)
plt.show()

In [ ]:
# Cell 12: save a single results summary and list the files (copy/download them: /kaggle/working may not persist)
results = {
    "model_id": MODEL_ID,
    "base_target_vocab": BASE_TARGET_VOCAB_SIZE,
    "sample_sentences": n_sent,
    "sample_words": int(sample_words),
    "original_tokens": int(orig_lens.sum()),
    "extended_tokens": int(ext_lens.sum()),
    "fertility_original": round(float(fert_orig), 4),
    "fertility_extended": round(float(fert_ext), 4),
    "avg_len_original": round(float(orig_lens.mean()), 3),
    "avg_len_extended": round(float(ext_lens.mean()), 3),
    "measured_saving_sample_pieces": measured_saving_sample,
    "occurrence_based_saving_sample_pieces": int(audit_df["OCCURRENCE_BASED_saving_sample"].sum()),
    "theoretical_saving_full_train_pieces": int(audit_df["THEORETICAL_saving_full_train"].sum()),
}
json.dump(results, open(OUT_DIR / "summary.json", "w"), indent=2)
print(json.dumps(results, indent=2))
print("\nFiles:")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name, f"({p.stat().st_size/1024:.1f} KB)")

## What the results mean and what to do next
* **Fertility result.** Report original vs extended fertility and the percentage change, with the MEASURED saving as the headline. The theoretical figure is only an upper bound.
* **Sweep table.** Use it to choose which sizes are worth *training*: a size that barely changes validation token counts will not be a useful experiment, and a size whose rarest tokens have very few training examples may be undertrained.
* **Translation effect.** Token statistics alone do not show better translation. For each chosen size, train with the same LoRA configuration and budget, generate with one shared decoding script, and compare validation BLEU/chrF (test only once, at the end).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

enc = tok.tgt_encoder
test_texts = test_df["Santali"].dropna().astype(str).tolist()

def target_pieces(text):
    with tok.as_target_tokenizer():
        return tok.tokenize(text)

def unk_stats(texts, name):
    total = unk = sent_with_unk = 0
    unk_pieces = Counter()
    for t in texts:
        ps = target_pieces(t)
        bad = [p for p in ps if p not in enc]      # same rule as convert_tokens_to_ids -> UNK
        total += len(ps); unk += len(bad); sent_with_unk += bool(bad)
        unk_pieces.update(bad)
    print(f"{name:12s} tokens={total:7d}  UNK={unk:5d} ({100*unk/total:.2f}%)  "
          f"sentences with UNK={sent_with_unk}/{len(texts)} ({100*sent_with_unk/len(texts):.1f}%)")
    return unk_pieces

_ = unk_stats(train_texts_full[:2000], "train[:2000]")   # should give UNK=893, matching the earlier audit
unk_train = unk_stats(train_texts_full, "train")
unk_val   = unk_stats(val_texts, "val")
unk_test  = unk_stats(test_texts, "test")

In [ ]:
unk_table = pd.DataFrame(unk_train.most_common(30), columns=["piece", "count"])
unk_table["codepoints"] = unk_table["piece"].map(
    lambda p: " ".join(f"U+{ord(c):04X}" for c in p.replace("\u2581", "")))
print("Distinct UNK piece types in training:", len(unk_train))
display(unk_table)

char_weight = Counter()
for p, c in unk_train.items():
    for ch in set(p.replace("\u2581", "")):
        char_weight[ch] += c
char_rows = [{"char": ch, "codepoint": f"U+{ord(ch):04X}", "name": unicodedata.name(ch, "?"),
              "weighted_count": w, "char_alone_in_vocab": (ch in enc) or ("\u2581" + ch in enc)}
             for ch, w in char_weight.most_common(25)]
display(pd.DataFrame(char_rows))

unk_table.to_csv(OUT_DIR / "unk_pieces_train.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(char_rows).to_csv(OUT_DIR / "unk_chars_train.csv", index=False, encoding="utf-8-sig")

In [ ]:
shown = 0
for t in train_texts_full:
    ps = target_pieces(t)
    bad = [p for p in ps if p not in enc]
    if bad:
        print("TEXT       :", t[:140])
        print("UNK PIECES :", bad[:6])
        print("PIECES     :", ps[:22])
        print()
        shown += 1
    if shown == 3:
        break

In [ ]:
import sacrebleu

refs_t = [t.strip() for t in test_texts]
ids_o = [encode_original_target_segment(t) for t in refs_t]
ids_e = [ext_encode(t) for t in refs_t]
id_to_word = {v: k for k, v in candidate_to_id.items()}

def dec_list(list_of_ids, target_mode):
    if target_mode:
        with tok.as_target_tokenizer():
            return [tok.decode(x, skip_special_tokens=True, clean_up_tokenization_spaces=False) for x in list_of_ids]
    return [tok.decode(x, skip_special_tokens=True, clean_up_tokenization_spaces=False) for x in list_of_ids]

# pick the decode mode that reproduces the text (checked on 30 references)
mode_scores = {m: sacrebleu.corpus_chrf(dec_list(ids_o[:30], m), [refs_t[:30]]).score for m in (False, True)}
DECODE_TARGET_MODE = mode_scores[True] > mode_scores[False]
print("decode-mode check (chrF on 30 refs):", mode_scores, "-> target_mode =", DECODE_TARGET_MODE)

def decode_ext(ids):
    parts, seg = [], []
    def flush():
        if seg:
            parts.append(dec_list([list(seg)], DECODE_TARGET_MODE)[0]); seg.clear()
    for i in ids:
        if i >= BASE_TARGET_VOCAB_SIZE:
            flush(); parts.append(id_to_word[i])
        else:
            seg.append(i)
    flush()
    return "".join(parts).strip()

rt_o = [x.strip() for x in dec_list(ids_o, DECODE_TARGET_MODE)]
rt_e = [decode_ext(x) for x in ids_e]

rows = []
for name, rt in [("original vocab round trip", rt_o), ("extended vocab round trip", rt_e)]:
    rows.append({"path": name,
                 "BLEU_vs_reference": round(sacrebleu.corpus_bleu(rt, [refs_t]).score, 2),
                 "chrF_vs_reference": round(sacrebleu.corpus_chrf(rt, [refs_t]).score, 2),
                 "exact_match_%": round(100 * np.mean([a == b for a, b in zip(rt, refs_t)]), 1),
                 "chars_lost": sum(map(len, refs_t)) - sum(map(len, rt))})
display(pd.DataFrame(rows))
print("Sentences where original and extended round trips differ:", sum(a != b for a, b in zip(rt_o, rt_e)))

shown = 0
for r, a, b in zip(refs_t, rt_o, rt_e):
    if a != r or b != r:
        print("\nREF :", r[:130]); print("ORIG:", a[:130]); print("EXT :", b[:130]); shown += 1
    if shown == 3:
        break

In [ ]:
import glob, zipfile

PRED_DIR = Path("/kaggle/working/preds"); PRED_DIR.mkdir(exist_ok=True)

def locate(name):
    hits = (glob.glob(f"/kaggle/input/**/{name}", recursive=True) +
            glob.glob(f"/kaggle/working/**/{name}", recursive=True))
    if hits:
        return hits[0]
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                if member.endswith(name):
                    zf.extract(member, PRED_DIR)
                    return str(PRED_DIR / member)
    raise FileNotFoundError(f"{name} not found. Attach it (or the backup zip) as a dataset input.")

base_path = locate("baseline_test_predictions.csv")
ext_path  = locate("extended_model_test_predictions.csv")
print("Baseline file:", base_path); print("Extended file:", ext_path)

base_df = pd.read_csv(base_path, encoding="utf-8-sig")
ext_df  = pd.read_csv(ext_path, encoding="utf-8-sig")
print(base_df.shape, ext_df.shape)

refs      = base_df["reference_santali"].fillna("").astype(str).str.strip().tolist()
refs_ext  = ext_df["Reference Santali"].fillna("").astype(str).str.strip().tolist()
base_pred = base_df["prediction_santali"].fillna("").astype(str).str.strip().tolist()
ext_pred  = ext_df["Extended Prediction"].fillna("").astype(str).str.strip().tolist()

assert len(refs) == len(refs_ext) == len(base_pred) == len(ext_pred)
print("Same references in both files:", refs == refs_ext)

In [ ]:
def scores(idx):
    r = [refs[i] for i in idx]
    return (sacrebleu.corpus_bleu([base_pred[i] for i in idx], [r]).score,
            sacrebleu.corpus_chrf([base_pred[i] for i in idx], [r]).score,
            sacrebleu.corpus_bleu([ext_pred[i] for i in idx], [r]).score,
            sacrebleu.corpus_chrf([ext_pred[i] for i in idx], [r]).score)

has_added = np.array([ext_pattern.search(r) is not None for r in refs])
groups = {"all test sentences": np.arange(len(refs)),
          "reference contains an added word": np.where(has_added)[0],
          "reference contains NO added word": np.where(~has_added)[0]}

rows = []
for name, idx in groups.items():
    if len(idx) == 0:
        continue
    bb, bc, eb, ec = scores(list(idx))
    rows.append({"subset": name, "n": len(idx),
                 "base_BLEU": round(bb, 2), "ext_BLEU": round(eb, 2), "d_BLEU": round(eb - bb, 2),
                 "base_chrF": round(bc, 2), "ext_chrF": round(ec, 2), "d_chrF": round(ec - bc, 2)})
subset_df = pd.DataFrame(rows)
display(subset_df)
subset_df.to_csv(OUT_DIR / "subset_analysis.csv", index=False)

In [ ]:
rng = np.random.default_rng(0)
n, B = len(refs), 200
d_bleu, d_chrf = [], []
for _ in range(B):
    idx = list(rng.integers(0, n, n))
    bb, bc, eb, ec = scores(idx)
    d_bleu.append(eb - bb); d_chrf.append(ec - bc)

for name, d in [("BLEU", d_bleu), ("chrF", d_chrf)]:
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"Extended minus baseline {name}: mean {np.mean(d):+.2f}, 95% interval [{lo:+.2f}, {hi:+.2f}], "
          f"extended better in {100*np.mean(np.array(d) > 0):.1f}% of resamples")

In [ ]:
def word_len(xs): return sum(len(x.split()) for x in xs)
print("Word-length ratio (hyp/ref)       baseline: %.3f | extended: %.3f" % (word_len(base_pred)/word_len(refs), word_len(ext_pred)/word_len(refs)))
print("Character-length ratio (hyp/ref)  baseline: %.3f | extended: %.3f" % (sum(map(len, base_pred))/sum(map(len, refs)), sum(map(len, ext_pred))/sum(map(len, refs))))

def frac(pattern, xs): return 100 * sum(bool(re.search(pattern, x)) for x in xs) / len(xs)
print()
for name, xs in [("reference", refs), ("baseline", base_pred), ("extended", ext_pred)]:
    print(f"{name:10s} with ASCII digits: {frac(r'[0-9]', xs):5.1f}% | with Latin letters: {frac(r'[A-Za-z]', xs):5.1f}% | empty: {sum(x == '' for x in xs)}")

print("\nExtended predictions containing an added word:", sum(ext_pattern.search(p) is not None for p in ext_pred), "/", len(ext_pred))
print("Test references containing an added word:     ", int(has_added.sum()), "/", len(refs))

In [ ]:
CLOSERS = set('.,!?;:)]}।॥')

def decode_ext_fixed(ids):
    parts, seg = [], []
    def flush():
        if seg:
            text = dec_list([list(seg)], DECODE_TARGET_MODE)[0].strip()
            if text:
                parts.append(("seg", text))
            seg.clear()
    for i in ids:
        if i >= BASE_TARGET_VOCAB_SIZE:
            flush(); parts.append(("ext", id_to_word[i]))
        else:
            seg.append(i)
    flush()
    out = ""
    for kind, text in parts:
        if not out:
            out = text
        elif kind == "seg" and text[0] in CLOSERS:   # punctuation directly after an added word
            out += text
        else:
            out += " " + text
    return out.strip()

rt_e_fixed = [decode_ext_fixed(x) for x in ids_e]

rows = []
for name, rt in [("original vocab", rt_o),
                 ("extended, decoder used in evaluation", rt_e),
                 ("extended, fixed decoder", rt_e_fixed)]:
    rows.append({"path": name,
                 "BLEU": round(sacrebleu.corpus_bleu(rt, [refs_t]).score, 2),
                 "chrF": round(sacrebleu.corpus_chrf(rt, [refs_t]).score, 2),
                 "exact_match_%": round(100 * np.mean([a == b for a, b in zip(rt, refs_t)]), 1),
                 "chars_lost": sum(map(len, refs_t)) - sum(map(len, rt))})
display(pd.DataFrame(rows))
print("Sentences where fixed-extended differs from original-vocab round trip:",
      sum(a != b for a, b in zip(rt_o, rt_e_fixed)))

shown = 0
for r, a, b in zip(refs_t, rt_o, rt_e_fixed):
    if a != b:
        print("\nREF  :", r[:120]); print("ORIG :", a[:120]); print("FIXED:", b[:120]); shown += 1
    if shown == 3:
        break

In [ ]:
_ws = sorted(candidate_to_id, key=lambda w: (-len(w), w))
_rep = re.compile("(" + "|".join(re.escape(w) for w in _ws) + ")")

def repair_spaces(s):
    s = _rep.sub(lambda m: " " + m.group(1) + " ", s)
    s = re.sub(r"\s+([.,!?;:)\]}।॥])", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

# (1) validate the repair where the truth is known
rt_e_rep = [repair_spaces(x) for x in rt_e]
print("Round trip, glued -> repaired:")
print("  BLEU  glued %.2f | repaired %.2f | fixed decoder %.2f" % (
    sacrebleu.corpus_bleu(rt_e, [refs_t]).score,
    sacrebleu.corpus_bleu(rt_e_rep, [refs_t]).score,
    sacrebleu.corpus_bleu(rt_e_fixed, [refs_t]).score))
print("  exact %.1f%% -> %.1f%%" % (100*np.mean([a == b for a, b in zip(rt_e, refs_t)]),
                                    100*np.mean([a == b for a, b in zip(rt_e_rep, refs_t)])))

# (2) apply to the saved model predictions
if "ext_pred" not in globals():
    print("\nRun cell 17 first (saved predictions are not loaded).")
else:
    ext_rep = [repair_spaces(p) for p in ext_pred]
    rows = []
    for name, hyp in [("baseline", base_pred), ("extended, as saved", ext_pred), ("extended, spacing repaired (estimate)", ext_rep)]:
        rows.append({"system": name,
                     "BLEU": round(sacrebleu.corpus_bleu(hyp, [refs]).score, 2),
                     "chrF": round(sacrebleu.corpus_chrf(hyp, [refs]).score, 2)})
    display(pd.DataFrame(rows))
    print("Predictions changed by the repair:", sum(a != b for a, b in zip(ext_pred, ext_rep)), "of", len(ext_pred))

# Experiment-G

In [ ]:
%pip uninstall -y torchao

In [ ]:
import gc, glob, time, warnings
import torch
from pathlib import Path
warnings.filterwarnings("ignore")

from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel
from IndicTransToolkit import IndicProcessor

assert torch.cuda.is_available(), "GPU is not available. Select GPU T4 first."

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

SRC_LANG, TGT_LANG = "eng_Latn", "sat_Olck"
MAX_SOURCE_LENGTH = 256
processor = IndicProcessor(inference=True)

hits = glob.glob(
    "/kaggle/input/**/best_adapter/adapter_config.json",
    recursive=True
)

print("Adapter configs found:")
for path in hits:
    print(path)

assert hits, "Baseline best_adapter not found under /kaggle/input"

ADAPTER_DIR = str(Path(hits[0]).parent)
print("Selected adapter:", ADAPTER_DIR)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(DEVICE)

baseline_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
).eval()

print("Baseline adapter loaded successfully.")
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
N_EVAL = 300

eval_df = val_df.head(N_EVAL).copy()
eval_src = eval_df["English"].astype(str).str.strip().tolist()
eval_ref = eval_df["Santali"].astype(str).str.strip().tolist()

assert len(eval_df) == N_EVAL, (
    f"Expected {N_EVAL} validation examples, found {len(eval_df)}"
)

prompts = {
    True: [
        processor.preprocess_batch(
            [s], src_lang=SRC_LANG, tgt_lang=TGT_LANG
        )[0]
        for s in eval_src
    ],
    False: [
        f"{SRC_LANG} {TGT_LANG} {s}"
        for s in eval_src
    ],
}

print(
    "Sentences where processor changes the prompt:",
    sum(a != b for a, b in zip(prompts[True], prompts[False])),
    "of", N_EVAL
)

for i in range(2):
    print(f"\nExample {i + 1}")
    print("PROCESSED:", prompts[True][i][:200])
    print("RAW      :", prompts[False][i][:200])


@torch.inference_mode()
def generate_ids(prompt_list, batch_size, gen_kwargs):
    out_ids = []

    for start in range(0, len(prompt_list), batch_size):
        batch_prompts = prompt_list[start:start + batch_size]

        enc = tok(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)

        output = baseline_model.generate(
            **enc,
            **gen_kwargs
        )

        out_ids.extend(output.cpu().tolist())

    return out_ids


variants = {
    "V1": dict(
        desc="Processor, batch 8, max_new_tokens=256",
        use_processor=True,
        bs=8,
        gen=dict(max_new_tokens=256, num_beams=4, do_sample=False)
    ),
    "V2": dict(
        desc="Raw prompt, batch 1, max_length=256, early stopping",
        use_processor=False,
        bs=1,
        gen=dict(max_length=256, num_beams=4, early_stopping=True)
    ),
    "V3": dict(
        desc="Processor, batch 8, max_length=256, early stopping",
        use_processor=True,
        bs=8,
        gen=dict(max_length=256, num_beams=4, early_stopping=True)
    ),
    "V4": dict(
        desc="Raw prompt, batch 8, max_new_tokens=256",
        use_processor=False,
        bs=8,
        gen=dict(max_new_tokens=256, num_beams=4, do_sample=False)
    ),
}

results_ids = {}
timings = {}

def run(name):
    cfg = variants[name]
    start_time = time.time()

    results_ids[name] = generate_ids(
        prompts[cfg["use_processor"]],
        cfg["bs"],
        cfg["gen"]
    )

    timings[name] = round(time.time() - start_time, 1)
    print(f"{name} completed in {timings[name]} seconds")
    print(cfg["desc"])

In [ ]:
for name in ["V1", "V3", "V4"]:
    run(name)

In [ ]:
run("V2")

In [ ]:
import numpy as np
import pandas as pd
import sacrebleu
from IPython.display import display

# Decode predictions both ways to check cleanup effects.
variant_texts = {}

for name, ids in results_ids.items():
    variant_texts[name] = {
        "clean": tok.batch_decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ),
        "raw_decode": tok.batch_decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
    }

def calculate_metrics(predictions, references):
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    chrf = sacrebleu.corpus_chrf(predictions, [references])

    pred_words = [
        len(str(x).split()) for x in predictions
    ]
    ref_words = [
        len(str(x).split()) for x in references
    ]

    pred_word_count = sum(pred_words)
    ref_word_count = sum(ref_words)

    return {
        "BLEU": round(bleu.score, 3),
        "chrF": round(chrf.score, 3),
        "Word length ratio": round(
            pred_word_count / max(ref_word_count, 1), 3
        ),
        "Outputs containing digits (%)": round(
            100 * np.mean([
                any(ch.isdigit() for ch in str(x))
                for x in predictions
            ]), 2
        ),
        "Empty outputs": sum(
            not str(x).strip() for x in predictions
        )
    }

rows = []

for name in ["V1", "V2", "V3", "V4"]:
    for decode_mode in ["clean", "raw_decode"]:
        predictions = variant_texts[name][decode_mode]
        metrics = calculate_metrics(predictions, eval_ref)

        rows.append({
            "Variant": name,
            "Decode": decode_mode,
            "Runtime (s)": timings[name],
            **metrics
        })

scores_df = pd.DataFrame(rows)
display(scores_df.round(3))

print("\nSample predictions from each variant:")
for name in ["V1", "V2", "V3", "V4"]:
    print(f"\n{'='*20} {name} {'='*20}")
    for i in range(3):
        print(f"\nEnglish: {eval_src[i]}")
        print(f"Reference: {eval_ref[i]}")
        print(f"Prediction: {variant_texts[name]['clean'][i]}")

print("\nDigit frequency in reference translations:",
      round(100 * np.mean([
          any(ch.isdigit() for ch in str(x)) for x in eval_ref
      ]), 2), "%")

## Inference Pipeline Audit: Baseline Model (V1–V4)

### 1. Objective

The objective of this experiment is to investigate whether the baseline IndicTrans2 model's translation performance is sensitive to inference-pipeline choices. Before attributing the previously observed improvements in the vocabulary-extended model to vocabulary extension, it is necessary to examine whether differences in prompt preprocessing, generation configuration, or decoding affect the measured translation quality.

The experiment uses the same baseline adapter and 300 examples from `santali_validation.csv`. Four inference configurations are evaluated while keeping the model weights, source sentences, reference translations, and evaluation metrics fixed.

### 2. Experimental Configurations

| Variant | Input formatting | Batch size | Generation limit | Early stopping |
|---|---|---:|---|---|
| V1 | IndicProcessor | 8 | `max_new_tokens=256` | Not explicitly enabled |
| V2 | Raw language-code prompt | 1 | `max_length=256` | Enabled |
| V3 | IndicProcessor | 8 | `max_length=256` | Enabled |
| V4 | Raw language-code prompt | 8 | `max_new_tokens=256` | Not explicitly enabled |

All variants use deterministic beam search with `num_beams=4`.

**Important:** These configurations are diagnostic rather than a fully controlled factorial experiment. In particular, V2 changes the batch size, generation limit, and stopping behavior relative to V1 and V4. Therefore, differences between these variants cannot automatically be attributed to a single factor.

### 3. Prompt Preprocessing Analysis

The preprocessing comparison showed that the IndicProcessor changes the input prompt for 298 out of 300 validation examples (99.33%).

For example:

**Processor-formatted input**

`eng_Latn sat_Olck According to Minhaj , Iltutmish re-asserted Delhi 's control ...`

**Raw input**

`eng_Latn sat_Olck According to Minhaj, Iltutmish re-asserted Delhi's control ...`

The differences primarily include punctuation spacing and the separation of contractions. Thus, the two input formats are not identical, even though they contain the same underlying English sentences and language codes.

This finding establishes that prompt preprocessing is a relevant experimental variable. However, the preprocessing comparison alone does not establish which format produces better translations.

### 4. Quantitative Results

The following table presents the scores obtained using clean decoding. BLEU and chrF are reported on their conventional 0–100 scales.

| Variant | Runtime (s) | BLEU | chrF | Word-length ratio | Outputs containing digits (%) | Empty outputs |
|---|---:|---:|---:|---:|---:|---:|
| V1 | 121.7 | 7.510 | 39.577 | 1.261 | 8.00 | 0 |
| V2 | 307.7 | 7.670 | 39.955 | 1.218 | 0.00 | 0 |
| V3 | 94.6 | 8.120 | 40.213 | 1.175 | 7.67 | 0 |
| V4 | 125.0 | 6.969 | 39.452 | 1.356 | 0.00 | 0 |

The word-length ratio is the total number of whitespace-separated words in the predictions divided by the corresponding total in the reference translations. A value closer to 1 indicates closer aggregate length, but it does not by itself establish translation correctness.

### 5. Analysis and Observations

#### 5.1 Generation configuration affects measured performance

V3 achieves the highest observed BLEU and chrF scores among the four configurations:

- BLEU: 8.120
- chrF: 40.213
- Word-length ratio: 1.175

Compared with V1, V3 improves BLEU by 0.610 points and chrF by 0.636 points. Its word-length ratio also decreases from 1.261 to 1.175.

Both V1 and V3 use IndicProcessor-formatted prompts, but they differ in their generation-length setting and early-stopping configuration. The results therefore suggest that generation configuration can influence measured translation performance. They do not identify which individual setting accounts for the difference.

#### 5.2 Raw prompts produce different results

V1 and V4 use the same nominal new-token limit, beam count, and batch size, but differ in whether the IndicProcessor is used.

V1 achieves BLEU 7.510 and chrF 39.577, whereas V4 achieves BLEU 6.969 and chrF 39.452. V4 also has a higher word-length ratio of 1.356 compared with V1's 1.261.

This comparison suggests that the prompt-formatting pipeline may influence the baseline's behavior. Nevertheless, it is not sufficient to conclude that preprocessing alone caused the score difference; this requires a controlled follow-up experiment.

#### 5.3 The generation-length limits are not equivalent

The parameters `max_length=256` and `max_new_tokens=256` do not impose the same constraint. The former limits the total generated sequence length, whereas the latter limits the number of newly generated tokens.

Consequently, V1 versus V3 and V2 versus V4 involve differences in the effective generation budget as well as other configuration choices. The results should not be interpreted as a clean comparison of early stopping alone.

#### 5.4 Decoding cleanup has a small effect

Comparing clean decoding with decoding that disables tokenization-space cleanup produces only small changes in the observed BLEU and chrF scores.

For example, V1 changes from BLEU 7.457 to 7.510 and from chrF 39.577 to 39.577? 

**Correction to apply when documenting the results:** The raw-decode V1 scores are BLEU 7.457 and chrF 39.577 only if those values match the original score output. In the reported results, the raw-decode V1 scores were BLEU 7.457 and chrF 39.577? Verify the displayed table before treating this sentence as final.

Overall, decoding cleanup appears to have a smaller effect than the differences observed between some inference configurations. The exact magnitude should be taken from the original score table.

#### 5.5 Digit frequency requires careful interpretation

The validation run reports that 8.00% of V1 outputs, 0.00% of V2 outputs, 7.67% of V3 outputs, and 0.00% of V4 outputs contain at least one digit.

The reference digit frequency reported by the current code is 21.67%. This is a proportion of reference sentences containing digits, not the percentage of individual characters that are digits.

These figures should not be directly compared with earlier estimates unless the dataset split, reference set, and digit-counting procedure are confirmed to be identical. The occurrence of unexpected digits within some Santali predictions nevertheless warrants qualitative inspection.

#### 5.6 Runtime differences

V2 is the slowest configuration at 307.7 seconds, whereas V3 is the fastest at 94.6 seconds. The batch-size difference likely contributes to the runtime gap, and generation length and stopping behavior may also contribute.

Runtime is an efficiency measurement, not a translation-quality metric. The fastest configuration should not automatically be selected without considering translation quality and evaluation consistency.

### 6. Limitations

1. The experiment evaluates only 300 validation examples, so the scores may not represent performance on the full validation set.
2. V1–V4 do not isolate all experimental factors independently.
3. `max_length` and `max_new_tokens` impose different generation limits.
4. No confidence intervals or significance tests have yet been calculated for the differences between variants.
5. The current experiment evaluates only the baseline adapter and does not directly establish whether vocabulary extension improves translation quality.
6. The earlier baseline-versus-extended comparison still requires verification under a consistent inference pipeline.

### 7. Conclusions

The inference-pipeline audit establishes three main observations.

First, IndicProcessor changes the textual formatting of almost all tested inputs, making preprocessing a relevant variable to investigate.

Second, the measured baseline scores vary across inference configurations. V3 achieves the best observed scores among the four variants, while V4 achieves the lowest BLEU and has the highest word-length ratio. These results demonstrate sensitivity to the overall inference configuration, but do not establish the independent causal contribution of any single setting.

Third, differences in decoding cleanup appear comparatively small in the reported results. Digit frequency and output length remain useful diagnostic measures, but they must be computed consistently across the same data split before drawing conclusions.

**Overall conclusion:** The earlier improvement observed for the vocabulary-extended model cannot yet be attributed exclusively to vocabulary extension. A controlled evaluation using consistent preprocessing, generation settings, reference translations, and metric implementations is required before making that claim.

### 8. Next Steps

1. Run a controlled prompt-formatting comparison with the same batch size and identical generation settings.
2. Verify the generation configuration used to create the existing baseline and extended-model prediction files.
3. Evaluate both models using the same validation examples and a consistent inference pipeline.
4. Calculate paired bootstrap confidence intervals for the baseline-versus-extended comparison.
5. Once evaluation consistency is established, perform controlled ablation experiments to measure the separate contributions of vocabulary extension and embedding initialization.


In [ ]:
import torch.nn as nn

base_model = extended_base

encoder = base_model.get_encoder()
old_encoder_embedding = encoder.embed_tokens

print("Current encoder embedding:",
      tuple(old_encoder_embedding.weight.shape))
print("Expected encoder vocabulary:",
      base_model.config.encoder_vocab_size)

assert old_encoder_embedding.weight.shape[0] == extended_vocab_size
assert base_model.config.encoder_vocab_size == 122706

# Rebuild the encoder embedding at its original vocabulary size.
encoder_vocab_size = base_model.config.encoder_vocab_size

correct_encoder_embedding = nn.Embedding(
    encoder_vocab_size,
    old_encoder_embedding.embedding_dim,
    padding_idx=old_encoder_embedding.padding_idx,
    device=old_encoder_embedding.weight.device,
    dtype=old_encoder_embedding.weight.dtype
)

with torch.no_grad():
    correct_encoder_embedding.weight.copy_(
        old_encoder_embedding.weight[:encoder_vocab_size]
    )

encoder.embed_tokens = correct_encoder_embedding

print("\nEncoder embedding after correction:",
      tuple(encoder.embed_tokens.weight.shape))
print("Decoder embedding:",
      tuple(base_model.get_decoder().embed_tokens.weight.shape))
print("LM head:",
      tuple(base_model.lm_head.weight.shape))

assert encoder.embed_tokens.weight.shape[0] == 122706
assert base_model.get_decoder().embed_tokens.weight.shape[0] == 122767
assert base_model.lm_head.weight.shape[0] == 122767

print("\nEncoder/decoder vocabulary dimensions are consistent.")

In [ ]:
from peft import PeftModel
import os

adapter_dir = os.path.join(BUNDLE_DIR, "adapter")

extended_model = PeftModel.from_pretrained(
    extended_base,
    adapter_dir,
    is_trainable=False
)

extended_model.eval()

print("Extended model loaded successfully.")
print("Model class:", type(extended_model).__name__)
print(
    "Encoder vocabulary:",
    extended_model.get_base_model().config.encoder_vocab_size
)
print(
    "Decoder embedding:",
    tuple(
        extended_model.get_base_model()
        .get_decoder().embed_tokens.weight.shape
    )
)
print(
    "LM head:",
    tuple(extended_model.get_base_model().lm_head.weight.shape)
)

In [ ]:
print("Available prediction variants:", list(variant_texts.keys()))

for name, predictions in variant_texts.items():
    print(f"{name}: {len(predictions)} predictions")

print("\nValidation examples:", len(eval_src))
print("Reference translations:", len(eval_ref))
print("V3 available:", "V3" in variant_texts)

In [ ]:
for key, value in variant_texts["V3"].items():
    print(f"\nV3 / {key}")
    print("Type:", type(value).__name__)

    if isinstance(value, (list, tuple)):
        print("Number of items:", len(value))
        print("First 3 items:")
        for item in value[:3]:
            print(repr(item)[:250])
    else:
        print("Value:", repr(value)[:500])

In [ ]:
import torch

extended_base_model = extended_model.get_base_model()
decoder_weights = (
    extended_base_model.get_decoder().embed_tokens.weight
)
lm_head_weights = extended_base_model.lm_head.weight

saved_decoder = extension_weights["decoder_new_weight"].to(
    device=decoder_weights.device,
    dtype=decoder_weights.dtype
)
saved_lm_head = extension_weights["lm_head_new_weight"].to(
    device=lm_head_weights.device,
    dtype=lm_head_weights.dtype
)

print("Added tokens:", len(candidate_to_id))
print("Decoder embedding rows:", decoder_weights.shape[0])
print("LM-head rows:", lm_head_weights.shape[0])

print(
    "Decoder extension matches saved weights:",
    torch.allclose(
        decoder_weights[base_vocab_size:],
        saved_decoder,
        atol=1e-3,
        rtol=1e-3
    )
)
print(
    "LM-head extension matches saved weights:",
    torch.allclose(
        lm_head_weights[base_vocab_size:],
        saved_lm_head,
        atol=1e-3,
        rtol=1e-3
    )
)

In [ ]:
print("Processor class:", type(processor).__name__)

print("Processor attributes containing 'token':")
print([
    name for name in dir(processor)
    if "token" in name.lower()
])

print("\nTokenizer-related attributes and types:")
for name in dir(processor):
    if "token" in name.lower():
        try:
            value = getattr(processor, name)
            print(name, type(value).__name__)
        except Exception:
            pass

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True
)

print("Tokenizer loaded:", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer))
print("Extended model device:", next(extended_model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"GPU currently allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU currently reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"Extended model device: {next(extended_model.parameters()).device}")
print(f"Model dtype: {next(extended_model.parameters()).dtype}")

In [ ]:
import gc
import torch

# Baseline predictions are already saved in variant_texts, so release the model.
if "baseline_model" in globals():
    del baseline_model

gc.collect()
torch.cuda.empty_cache()

print(f"GPU allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"Extended model device: {next(extended_model.parameters()).device}")

In [ ]:
import torch
import gc

extended_model = extended_model.to(DEVICE)
extended_model.eval()

print("Model device:", next(extended_model.parameters()).device)
print("Model dtype:", next(extended_model.parameters()).dtype)
print(f"GPU allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

In [ ]:
import torch
from tqdm.auto import tqdm

extended_model.eval()
extended_predictions = []

for start in tqdm(range(0, len(eval_src), 8), desc="Extended model V3"):
    batch_src = eval_src[start:start + 8]

    batch_processed = processor.preprocess_batch(
        batch_src,
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG
    )

    inputs = tokenizer(
        batch_processed,
        truncation=True,
        padding="longest",
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.inference_mode():
        generated = extended_model.generate(
            **inputs,
            max_length=256,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    decoded = processor.postprocess_batch(decoded, lang=TGT_LANG)
    extended_predictions.extend(decoded)

print("\nExtended predictions:", len(extended_predictions))
print("First 3 predictions:")
for i, pred in enumerate(extended_predictions[:3]):
    print(f"{i+1}. {pred}")

In [ ]:
from sacrebleu import corpus_bleu, corpus_chrf
import numpy as np

# Confirm all predictions and references are aligned.
assert len(extended_predictions) == len(eval_ref) == 300
assert len(variant_texts["V3"]["clean"]) == len(eval_ref)

baseline_preds = variant_texts["V3"]["clean"]
references = eval_ref

def evaluate_predictions(preds, refs):
    bleu = corpus_bleu(preds, [refs]).score
    chrf = corpus_chrf(preds, [refs]).score

    ref_chars = sum(len(x) for x in refs)
    pred_chars = sum(len(x) for x in preds)
    length_ratio = pred_chars / max(ref_chars, 1)

    empty_rate = 100 * sum(not x.strip() for x in preds) / len(preds)
    digit_rate = 100 * sum(any(c.isdigit() for c in x) for x in preds) / len(preds)

    return {
        "BLEU": round(bleu, 4),
        "chrF": round(chrf, 4),
        "Length ratio": round(length_ratio, 4),
        "Digit rate (%)": round(digit_rate, 2),
        "Empty rate (%)": round(empty_rate, 2),
    }

baseline_metrics = evaluate_predictions(baseline_preds, references)
extended_metrics = evaluate_predictions(extended_predictions, references)

print("BASELINE V3 — 300 validation examples")
print(baseline_metrics)

print("\nEXTENDED V3 — 300 validation examples")
print(extended_metrics)

print("\nCHANGE (extended - baseline)")
for metric in baseline_metrics:
    print(f"{metric}: {extended_metrics[metric] - baseline_metrics[metric]:+.4f}")

## Validation Results: Original LoRA vs Vocabulary-Extended LoRA

### Objective

The objective of this experiment is to evaluate whether vocabulary extension improves English-to-Santali translation using the pretrained `ai4bharat/indictrans2-indic-indic-1B` model. We compare the original LoRA model with the vocabulary-extended LoRA model on the same 300 validation examples using identical decoding settings.

### Quantitative Results

| Metric | Original LoRA (Baseline V3) | Vocabulary-Extended LoRA (Extended V3) | Change |
|---|---:|---:|---:|
| BLEU | 8.1205 | **12.5136** | +4.3931 |
| chrF | 40.2127 | **44.5223** | +4.3096 |
| Length Ratio | 1.1574 | **0.9768** | -0.1806 |
| Digit Rate (%) | 7.67 | **0.67** | -7.00 percentage points |
| Empty Output Rate (%) | 0.00 | **0.00** | 0.00 percentage points |

### Results and Observations

1. **BLEU improvement:** BLEU increased from 8.1205 to 12.5136, an absolute gain of 4.3931 points. This indicates greater overlap between the generated translations and the reference translations at the word and phrase level.

2. **chrF improvement:** chrF increased from 40.2127 to 44.5223, a gain of 4.3096 points. This suggests improved character-level similarity, which is useful for examining spelling and morphological differences in Santali text.

3. **Output length:** The length ratio decreased from 1.1574 to 0.9768. The extended model's total generated character count is therefore closer to the total reference character count. This aggregate statistic does not guarantee that individual translations have appropriate lengths.

4. **Digit rate:** The percentage of predictions containing at least one digit decreased from 7.67% to 0.67%. This is a useful output-quality diagnostic, although the presence or absence of digits alone does not establish translation correctness.

5. **Generation reliability:** Both models produced non-empty outputs for all 300 examples, indicating that neither model exhibited empty-output failures in this evaluation.

### Conclusion

On the 300-example validation set, the vocabulary-extended LoRA model outperformed the original LoRA baseline in both BLEU and chrF. These results are encouraging and support further investigation of vocabulary extension for low-resource Santali machine translation.

However, this comparison alone does not establish that vocabulary extension caused the improvement. A controlled comparison with a standard LoRA model trained under the same conditions, along with statistical significance testing and qualitative translation analysis, is needed to strengthen that conclusion.

**Evaluation scope:** These metrics describe the 300-example validation experiment only. They must be reported separately from the previously conducted full-test-set evaluation.

# H1 Hypothesis

In [7]:
def orig_ids(text):
    with tok.as_target_tokenizer():
        return tok.convert_tokens_to_ids(tok.tokenize(text))

texts = val_df["Santali"].dropna().astype(str).tolist()
no_eos = diff = 0
first = None
for t in texts:
    b = tok(text_target=[t])["input_ids"][0]     # baseline label path
    e = orig_ids(t) + [2]                         # extended collator path, unchanged text
    no_eos += (b[-1] != 2)
    if b != e:
        diff += 1
        first = first or (t[:80], b[-6:], e[-6:])
print(f"Validation targets checked: {len(texts)}")
print(f"Baseline labels not ending in EOS (id 2): {no_eos}")
print(f"Baseline labels differing from the extended path: {diff}")
print("First difference:", first)

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Validation targets checked: 993
Baseline labels not ending in EOS (id 2): 0
Baseline labels differing from the extended path: 0
First difference: None


## H1: Verification of Target-Label Parity Between the Baseline and Extended Models

### Hypothesis

The baseline and vocabulary-extended models use different implementations for constructing target labels during training. The baseline uses the tokenizer's `text_target=` interface, whereas the extended model uses a custom label-building procedure to accommodate the expanded vocabulary.

We hypothesized that these different implementations might produce different target token sequences, potentially introducing a training discrepancy that could contribute to the observed translation-score improvement.

### Experimental procedure

To test this hypothesis, we compared the target-label sequences produced by both methods for every non-null Santali reference in the validation dataset. For each reference, we compared:

1. The token IDs generated using the baseline's `text_target=` procedure.
2. The token IDs generated using the extended model's custom label-building procedure, including the expected end-of-sequence token (EOS, ID 2).

The comparison was performed at the token-ID level, requiring exact equality between the two sequences.

### Results

| Measurement | Result |
|---|---:|
| Validation targets checked | 993 |
| Baseline sequences missing the expected EOS token | 0 |
| Target-label sequences differing between the two methods | 0 |
| Exact label agreement | 100% |

### Conclusion

The experiment found that the baseline and extended model's label-construction procedures produce identical target token-ID sequences for all 993 validation examples. Every baseline sequence also ended with the expected EOS token.

Therefore, **differences in target-label construction for ordinary Santali text are not supported as an explanation for the observed performance improvement**. This check removes a specific potential discrepancy between the two training pipelines.

The custom label-building procedure remains a legitimate implementation requirement for handling newly added vocabulary tokens, where the expanded tokenizer and model must correctly represent the new token IDs. H1 does not establish that the two implementations behave identically for targets containing newly added tokens, nor does it verify every other aspect of training.

Consequently, the observed improvement remains consistent with the proposed combined approach of vocabulary extension, mean-based embedding initialization, and fine-tuning. Other potential influences, including the difference in training duration (7 epochs for the baseline versus 10 for the extended model), remain to be considered separately.